##  나만의 RAG 시스템 구축 실습문제

다음 단계에 따라 나만의 RAG 앱을 구현하세요.


#### 1단계: 데이터 수집 및 구축

* 관심 있는 주제를 정하고, 관련 정보를 수집한다.
* pdf를 추천하지만, Langchain Document화 할수 모든 데이터 가능하다.

#### 2단계: 임베딩 모델 선정 및 문서 임베딩

* 사용할 임베딩 모델을 선정한다.
  * 예: OpenAI, Hugging Face, 또는 자체 임베딩 모델
* 수집한 데이터를 임베딩 벡터로 변환하여 저장한다.
  * 벡터DB(예: FAISS, Chroma, Pinecone 등) 사용 가능

#### 3단계: 검색(Retrieval) & 생성(Generation) 파이프라인 구현

* 사용자의 질의(Query)를 임베딩하고, 벡터DB에서 가장 관련 있는 문서를 검색하는 기능을 구현한다.
* 검색된 문서를 LLM에 입력해 자연어로 답변을 생성하는 기능을 만든다.

#### 4단계: End-to-End 통합 및 테스트

* 전체 파이프라인을 하나의 함수/시스템으로 통합한다.
* 실제로 여러 질문을 입력해 답변이 잘 나오는지 확인한다.

## 환경 설정

In [20]:
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
# PINECONE_INDEX_NAME = 'adv-rag'
PINECONE_META_INDEX_NAME = 'adv-meta-rag'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_METRIc = 'cosine'
PINECONE_INDEX_DEMENSION = 1536

OPENAI_LLM_MODEL = 'gpt-4.1-mini'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'

## 데이터 로드

In [22]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('data/League_of_Legends_Operational_Policy_20260227.pdf')
docs = loader.load()
print(len(docs))

12


In [23]:
doc = docs[1]
print(doc.page_content)
print(doc.metadata)

② GM은 원활한 게임 진행을 돕고 본 운영정책에 위배되는 행위를 근절하여 플레이어에게 즐거운 게임 환경을 제공하기
위해 최선의 노력을 다합니다.
③ GM은 플레이어의 문의에 대해서 최대한 신속히 답변하고, 이를 공정하게 처리하기 위해 노력합니다.
④ GM은 사사로운 이익, 친분 관계 또는 개인적인 감정으로 문제를 판단하지 않으며, 항상 객관적으로 상황을 파악하고
이성적으로 판단하여 적절한 절차 및 정책에 따라 모든 문제를 처리합니다.
⑤ GM은 게임 내에서 플레이어의 행동에 개입하지 않습니다. 다만, 원활한 게임 운영이나 즐겁고 건전한 게임 문화
조성을 위해 필요하다고 판단되는 경우, GM은 본 운영정책에 따라 강제 접속 종료, 이용제한, 모니터링 등의 조치를
취할 수 있습니다.
⑥ GM은 플레이어 계정의 비밀번호를 묻지 않으며, 업무상 취득한 플레이어 정보를 외부에 유출하지 않습니다. 단,
정부기관 또는 수사기관 등에서 적법한 절차에 따라 개인정보 등의 정보 제공을 요청한 경우에는 회사의 판단에 따라
해당 기관에 제한적으로 정보를 제공할 수 있습니다.
ㅤ
ㅤ
제3조. 플레이어의 권리와 의무
① 플레이어는 서비스 약관과 본 운영정책에 위배되는 행위를 지양해야 하며, 이를 위반할 경우 관련 정책에 따라
이용제한을 받을 수 있습니다. 또한, 지속적으로 이용제한을 받거나 그 위반행위의 정도가 매우 심각한 경우, 보다 상위
차수의 이용제한으로 수위가 강화될 수 있습니다. 특히, 중대한 위반행위의 경우, 해당 계정에서 게임의 이용이
영구적으로 제한되거나 동일 명의로 생성된 모든 계정에서 게임의 이용이 영구적으로 제한될 수도 있으며, 동일 명의
모든 계정에 대한 영구적인 이용제한에 더해 회원가입 제한이 발생할 수도 있습니다. 더불어, 서비스 약관에 근거하여,
사안에 따라 한 게임에서의 불건전 행위를 바탕으로 회사가 서비스를 제공하는 다른 모든 게임에서 상기 언급한 조치와
유사하거나 가중된 조치가 동시에 발생할 수 있습니다.
② 플레이어는 서비스 약관 및 본 운영정책 등

## 데이터 전처리

In [24]:
# 페이지 마지막 링크, 페이지 수 제거
import re

def clean_text(text):
    text = re.sub(
        r"https://legal\.kr\.riotgames\.com/league/service\s+\d+/\d+",
        "",
        text
    )
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [25]:
# metadata 정리
def clean_metadata(metadata):
    return {
        "source": metadata.get("source"),
        "title": metadata.get("title"),
        "page": metadata.get("page"),
        "page_label": metadata.get("page_label"),
        "total_pages": metadata.get("total_pages"),
        "url": "https://legal.kr.riotgames.com/league/service",
        "doc_type": "operational_policy",
        "service": "league_of_legends",
    }

In [26]:
from langchain_core.documents import Document

cleaned_docs = []

for doc in docs:
    cleaned_content = clean_text(doc.page_content)
    cleaned_meta = clean_metadata(doc.metadata)

    cleaned_docs.append(
        Document(
            page_content=cleaned_content,
            metadata=cleaned_meta
        )
    )

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 120,
    separators=["\n\n", "\n", "다.", " ", ""]
)

chunks = chunk_splitter.split_documents(cleaned_docs)

In [28]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./db/riot_policy"
)

In [29]:
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs={"k":4}
)

In [30]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model=OPENAI_LLM_MODEL
)

output_parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("""
너는 질문자에게 친절하게 이용 약관을 알려주는 법률 AI입니다.
                                          
규칙:
1. [Context]에 있는 내용만 근거로 사용하세요.
2. 질문에 직접 답하는 내용만 작성하세요.
3. 문서에 없는 내용은 추측하지 마세요.
4. 질문에 답할 근거가 부족하면 "제공된 문서만으로는 답변할 수 없습니다."라고 답하세요.
5. 답변에 사용한 문서 page를 문장 끝에 표시하세요.
6. 답변 마지막에 "참고 문서" 항목을 작성하세요. 
                                          
[Context]
{context}
                                            
[Question]
{question}
                                            
[Answer]
""")

In [31]:
def format_docs(docs):
    return "\n\n".join(
        f"[출처: page {doc.metadata.get('page_label')}]\n"
        f"{doc.page_content}"
        for doc in docs
    )

In [32]:
from langchain_core.runnables import RunnablePassthrough

chain = ({
    "context":retriever|format_docs,
    'question':RunnablePassthrough()
    }
    |prompt 
    | llm 
    | output_parser
)

In [34]:
def ask_rag(question):
    answer = chain.invoke(question)
    return answer

In [35]:
questions = [
    "영구정지를 하는 기준이 뭐야?",
    "욕설을 하면 어떤 제재를 받을 수 있어?",
    "탈주하면 제재를 받아?",
    "계정 공유는 금지되어 있어?",
    "채팅 제한은 어떤 경우에 받아?"
]

for question in questions:
    print(ask_rag(question))
    print('-'*100)

영구정지는 게임 진행 방해, 언어 폭력, 부적절한 이름 사용, 게임·계정 등 콘텐츠 거래와 같은 불건전 행위에 대해 적용될 수 있습니다. 이러한 행위가 심각할 경우, 해당 계정뿐만 아니라 동일 명의의 모든 계정에 대해 영구 이용제한이 내려질 수 있습니다. 또한, 영구 이용제한과 함께 회원가입 제한도 발생할 수 있으며, 한 게임에서의 불건전 행위로 인해 회사가 제공하는 다른 게임에서 유사하거나 가중된 조치가 동시에 이루어질 수 있습니다. (출처: page 2, 9)

참고 문서: page 2, 8, 9
----------------------------------------------------------------------------------------------------
욕설이 포함된 이름을 사용하는 경우, 이용제한이 발생할 수 있으며 심각한 불건전 행위로 판단될 경우 민사 및 형사상의 책임이 발생할 수 있습니다. 또한, 불건전 행위가 반복되거나 고의성이 있는 경우 더 높은 단계의 이용제한이 적용되며, 최상위 수준의 이용제한(동일 명의의 모든 계정 영구 이용제한 및 회원가입 제한)까지 받을 수 있습니다. 이는 욕설 또는 비속어 사용이 명예훼손 등 권리 침해와 관련되어 있기 때문입니다. (출처: page 3, 4, 8)

참고 문서: page 3, 4, 8
----------------------------------------------------------------------------------------------------
탈주 행위에 대한 구체적인 제재 내용은 제공된 문서에 명시되어 있지 않습니다. 따라서 탈주 시 제재 여부에 대해서는 제공된 문서만으로는 답변할 수 없습니다. (page 7-8)

참고 문서: page 7-8
----------------------------------------------------------------------------------------------------
제공된 문서만으로는 계정 공유 금지 여부